In [4]:
import requests
import minsearch
import ollama
from dotenv import load_dotenv
import re

load_dotenv()

print("All imports are working!")

All imports are working!


In [5]:
url="https://raw.githubusercontent.com/python/cpython/main/Doc/faq/general.rst"
response = requests.get(url)
if response.status_code == 200:
    print("Request to URL was successful!") 

Request to URL was successful!


In [6]:
print("length:", len(response.content))


length: 20130


In [7]:
print(response.text[1200:2000])

ndent non-profit organization that
holds the copyright on Python versions 2.1 and newer.  The PSF's mission is to
advance open source technology related to the Python programming language and to
publicize the use of Python.  The PSF's home page is at
https://www.python.org/psf/.

Donations to the PSF are tax-exempt in the US.  If you use Python and find it
helpful, please contribute via `the PSF donation page
<https://www.python.org/psf/donations/>`_.


Are there copyright restrictions on the use of Python?
------------------------------------------------------

You can do anything you want with the source, as long as you leave the
copyrights in and display those copyrights in any documentation about Python
that you produce.  If you honor the copyright rules, it's OK to use Python for
comm


In [ ]:
#i just checked the response whether it retieved raw or with html
faq_text = response.text
if faq_text.lstrip().lower().startswith("<!doctype html"):
    raise ValueError("Received HTML instead of the raw FAQ text.")


In [9]:
import re

def extract_faqs(text):
    lines = text.splitlines()
    documents = []

    i = 0

    while i < len(lines) - 1:
        title = lines[i].strip()
        underline = lines[i + 1].strip()

        # Find reStructuredText headings that are questions.
        is_question = (
            title.endswith("?")
            and len(underline) >= 3
            and len(set(underline)) == 1
            and underline[0] in "=-~^\"'`:+*#"
        )

        if is_question:
            question = title
            answer_lines = []
            i += 2

            while i < len(lines):
                current = lines[i].strip()

                # Stop when another heading begins.
                if (
                    i + 1 < len(lines)
                    and current
                    and len(lines[i + 1].strip()) >= 3
                    and len(set(lines[i + 1].strip())) == 1
                    and lines[i + 1].strip()[0] in "=-~^\"'`:+*#"
                ):
                    break

                answer_lines.append(lines[i])
                i += 1

            answer = "\n".join(answer_lines).strip()
            answer = re.sub(r"\n{3,}", "\n\n", answer)

            if answer:
                documents.append({
                    "question": question,
                    "answer": answer
                })
        else:
            i += 1

    return documents


documents = extract_faqs(faq_text)



In [10]:
print("Questions extracted:", len(documents))
print(documents[:3])

Questions extracted: 23
[{'question': 'What is Python?', 'answer': "Python is an interpreted, interactive, object-oriented programming language.  It\nincorporates modules, exceptions, dynamic typing, very high level dynamic data\ntypes, and classes.  It supports multiple programming paradigms beyond\nobject-oriented programming, such as procedural and functional programming.\nPython combines remarkable power with very clear syntax. It has interfaces to\nmany system calls and libraries, as well as to various window systems, and is\nextensible in C or C++.  It is also usable as an extension language for\napplications that need a programmable interface. Finally, Python is portable:\nit runs on many Unix variants including Linux and macOS, and on Windows.\n\nTo find out more, start with :ref:`tutorial-index`.  The `Beginner's Guide to\nPython <https://wiki.python.org/moin/BeginnersGuide>`_ links to other\nintroductory tutorials and resources for learning Python."}, {'question': 'What is th

In [11]:
#cleaning the extracted documents in qstion and answer, and saving them to a JSON file
import json
from pathlib import Path

clean_documents = []

for doc in documents:
    question = " ".join(doc["question"].split())
    answer = doc["answer"].strip()

    if question and answer:
        clean_documents.append({
            "question": question,
            "answer": answer
        })

if not clean_documents:
    raise ValueError(
        "No FAQ pairs extracted. Check the parser before continuing."
    )

Path("data").mkdir(exist_ok=True)

with open("data/faqs.json", "w", encoding="utf-8") as f:
    json.dump(clean_documents, f, ensure_ascii=False, indent=2)

print("Saved FAQ pairs:", len(clean_documents))
print("First cleaned document:")
print(json.dumps(clean_documents[0], indent=2))

Saved FAQ pairs: 23
First cleaned document:
{
  "question": "What is Python?",
  "answer": "Python is an interpreted, interactive, object-oriented programming language.  It\nincorporates modules, exceptions, dynamic typing, very high level dynamic data\ntypes, and classes.  It supports multiple programming paradigms beyond\nobject-oriented programming, such as procedural and functional programming.\nPython combines remarkable power with very clear syntax. It has interfaces to\nmany system calls and libraries, as well as to various window systems, and is\nextensible in C or C++.  It is also usable as an extension language for\napplications that need a programmable interface. Finally, Python is portable:\nit runs on many Unix variants including Linux and macOS, and on Windows.\n\nTo find out more, start with :ref:`tutorial-index`.  The `Beginner's Guide to\nPython <https://wiki.python.org/moin/BeginnersGuide>`_ links to other\nintroductory tutorials and resources for learning Python."
}


In [12]:
#Step 6 — Create the MinSearch index
from minsearch import Index

index = Index(
    text_fields=["question", "answer"]
)

index.fit(clean_documents)

print("MinSearch index created successfully!")

MinSearch index created successfully!


In [13]:
#Test retrieval before using Ollama
query = "How do I submit bug reports and patches for Python?"

results = index.search(
    query,
    num_results=3
)

print("Retrieved documents:", len(results))

for i, doc in enumerate(results, start=1):
    print(f"\nResult {i}")
    print("Question:", doc["question"])
    print("Answer:", doc["answer"][:500])

Retrieved documents: 3

Result 1
Question: How do I submit bug reports and patches for Python?
Answer: To report a bug or submit a patch, use the issue tracker at
https://github.com/python/cpython/issues.

For more information on how Python is developed, consult `the Python Developer's
Guide <https://devguide.python.org/>`_.

Result 2
Question: How do I get documentation on Python?
Answer: The standard documentation for the current stable version of Python is available
at https://docs.python.org/3/.  EPUB, plain text, and downloadable HTML versions are
also available at https://docs.python.org/3/download.html.

The documentation is written in reStructuredText and processed by `the Sphinx
documentation tool <https://www.sphinx-doc.org/>`__.  The reStructuredText source for
the documentation is part of the Python source distribution.

Result 3
Question: How do I get a beta test version of Python?
Answer: Alpha and beta releases are available from https://www.python.org/downloads/.  All
r

In [14]:
#connecting it to llm (ollama is used here)
import ollama

MODEL = "llama3.2:3b "

response = ollama.chat(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": "Explain Python in two simple sentences."
        }
    ]
)

print(response.message.content)

Python is a high-level programming language that allows developers to write clear, concise, and readable code for a wide range of applications, from web development to data analysis and artificial intelligence. Its simplicity, flexibility, and vast library of modules make it a popular choice for beginners and experienced programmers alike.


In [19]:
def ask_faq(question, top_k=3):
    # 1. Retrieve relevant FAQ documents.
    results = index.search(
        question,
        num_results=top_k
    )

    if not results:
        return "I couldn't find a relevant FAQ for that question."

    # 2. Build the context from retrieved documents.
    context = "\n\n".join(
        f"Question: {doc['question']}\n"
        f"Answer: {doc['answer']}"
        for doc in results
    )

    # 3. Ask Ollama to answer using that context.
    response = ollama.chat(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a Python FAQ assistant. "
                    "Answer using only the supplied FAQ context. "
                    "If the context does not contain the answer, "
                    "say you could not find the answer in the FAQs. "
                    "Do not invent information."
                )
            },
            {
                "role": "user",
                "content": (
                    f"FAQ context:\n{context}\n\n"
                    f"User question: {question}"
                )
            }
        ]
    )

    return response.message.content


print(ask_faq("What is Python used for?"))

Python is used for a wide variety of purposes, including:

* General-purpose programming
* Web development (with various frameworks and libraries)
* Data analysis and science
* Machine learning and artificial intelligence
* Automation and scripting
* Scientific computing
* Network programming
* Game development
* And many other areas.

It is a versatile language that can be applied to many different classes of problems, and its extensive standard library and third-party extensions make it a popular choice for a wide range of applications.


In [ ]:
#just for testing
results = index.search(
    "What is Python?",
    num_results=3
)

for i, doc in enumerate(results, start=1):
    print(f"\n--- Result {i} ---")
    print("Question:", doc["question"])
    print("Answer:", doc["answer"][:500])


--- Result 1 ---
Question: What is Python?
Answer: Python is an interpreted, interactive, object-oriented programming language.  It
incorporates modules, exceptions, dynamic typing, very high level dynamic data
types, and classes.  It supports multiple programming paradigms beyond
object-oriented programming, such as procedural and functional programming.
Python combines remarkable power with very clear syntax. It has interfaces to
many system calls and libraries, as well as to various window systems, and is
extensible in C or C++.  It is also u

--- Result 2 ---
Question: What is Python good for?
Answer: Python is a high-level general-purpose programming language that can be applied
to many different classes of problems.

The language comes with a large standard library that covers areas such as
string processing (regular expressions, Unicode, calculating differences between
files), internet protocols (HTTP, FTP, SMTP, XML-RPC, POP, IMAP),
software engineering (unit testing, logging,